In [ ]:
# CELL 1 | Install all required packages | Run ONCE then restart runtime

!apt-get update -qq
!apt-get install -y -qq libspatialindex-dev libgeos-dev

!pip install -qq \
    laspy[lazrs] \
    rasterio \
    geopandas \
    shapely \
    xgboost \
    scikit-learn \
    pyproj \
    whitebox \
    tqdm \
    cloth-simulation-filter \
    matplotlib \
    networkx

print("Done")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libspatialindex6:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libspatialindex6_1.9.3-2_amd64.deb ...
Unpacking libspatialindex6:amd64 (1.9.3-2) ...
Selecting previously unselected package libspatialindex-c6:amd64.
Preparing to unpack .../libspatialindex-c6_1.9.3-2_amd64.deb ...
Unpacking libspatialindex-c6:amd64 (1.9.3-2) ...
Selecting previously unselected package libspatialindex-dev:amd64.
Preparing to unpack .../libspatialindex-dev_1.9.3-2_amd64.deb ...
Unpacking libspatialindex-dev:amd64 (1.9.3-2) ...
Setting up libspatialindex6:amd64 (1.9.3-2) ...
Setting up libspatialindex-c6:amd64 (1.9.3-2) ...
Setting up libspatialindex-dev:amd64 (1.9.3-2) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) 

In [ ]:
# CELL 2 | Mount Google Drive and load all libraries

from google.colab import drive
drive.mount('/content/drive')
import os, shutil, gc, warnings, heapq
warnings.filterwarnings('ignore')
import numpy as np
from scipy.ndimage import (median_filter, gaussian_filter,
                            uniform_filter, minimum_filter,
                            laplace, label as scipy_label,
                            center_of_mass)
import laspy
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape, LineString
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
import networkx as nx
import CSF

print("All imports successful")
print(f"  NumPy    : {np.__version__}")
print(f"  Rasterio : {rasterio.__version__}")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  XGBoost  : {xgb.__version__}")

Mounted at /content/drive
All imports successful
  NumPy    : 2.0.2
  Rasterio : 1.5.0
  GeoPandas: 1.1.3
  XGBoost  : 3.2.0


In [ ]:
# CELL 3 | Output folders + all 10 villages registered

BASE     = '/content/drive/MyDrive/hackathon_data'
OUT_BASE = '/content/drive/MyDrive/mopr_outputs'

for f in ['ground','dtm','gpkg','rasters','reports','village_reports']:
    os.makedirs(f'{OUT_BASE}/{f}', exist_ok=True)
os.makedirs('/tmp/mopr', exist_ok=True)

NODATA  = -9999.0
MAX_PTS = 5_000_000

VILLAGES = {
    'DEVDI':           {'raw': f'{BASE}/DEVDI_POINT CLOUD (511671).las',        'epsg': 32643, 'res': 1.0},
    'KHAPRETA':        {'raw': f'{BASE}/KHAPRETA_510206.laz',                   'epsg': 32643, 'res': 1.0},
    'PIRAYANKUPPAM':   {'raw': f'{BASE}/PIRAYANKUPPAM.las',                     'epsg': 32644, 'res': 1.0},
    'DHUNDA':          {'raw': f'{BASE}/DHUNDA_FATEHGARH SAHIB_32619.laz',      'epsg': 32643, 'res': 1.0},
    'DHAL':            {'raw': f'{BASE}/Dhal_Hoshiarpur_31235.las',             'epsg': 32643, 'res': 1.0},
    'THANDALAM':       {'raw': f'{BASE}/THANDALAM.las',                         'epsg': 32644, 'res': 1.0},
    'REFLIGHT_64334':  {'raw': f'{BASE}/64334_2H (REFLIGHT)_POINT CLOUD.LAS',  'epsg': 32643, 'res': 1.0},
    'CHAKHIRASINGH':   {'raw': f'{BASE}/67169_5NKR_CHAKHIRASINGH.las',          'epsg': 32643, 'res': 1.0},
    'GANDHINAGAR_DIG': {'raw': f'{BASE}/Gandhinagar_Diglipur_group1_densified_point_cloud.laz', 'epsg': 32646, 'res': 1.0},
    'KADAMTALA_RNG':   {'raw': f'{BASE}/Kadamtala_Rangat_A&N_02022022_group1_densified_point_cloud.laz', 'epsg': 32646, 'res': 1.0},
}

# Andaman villages: Diglipur & Rangat are in UTM Zone 46N (EPSG:32646)
# REFLIGHT/CHAKHIRASINGH: auto-detect will override epsg if geographic

print(f"{'Village':<22} {'File':<60} {'Exists'}")
print("-" * 90)
for name, cfg in VILLAGES.items():
    exists = os.path.exists(cfg['raw'])
    print(f"  {name:<20} {cfg['raw'].split('/')[-1]:<58} {'OK' if exists else 'MISSING'}")

Village                File                                                         Exists
------------------------------------------------------------------------------------------
  DEVDI                DEVDI_POINT CLOUD (511671).las                             OK
  KHAPRETA             KHAPRETA_510206.laz                                        OK
  PIRAYANKUPPAM        PIRAYANKUPPAM.las                                          OK
  DHUNDA               DHUNDA_FATEHGARH SAHIB_32619.laz                           OK
  DHAL                 Dhal_Hoshiarpur_31235.las                                  OK
  THANDALAM            THANDALAM.las                                              OK
  REFLIGHT_64334       64334_2H (REFLIGHT)_POINT CLOUD.LAS                        OK
  CHAKHIRASINGH        67169_5NKR_CHAKHIRASINGH.las                               OK
  GANDHINAGAR_DIG      Gandhinagar_Diglipur_group1_densified_point_cloud.laz      OK
  KADAMTALA_RNG        Kadamtala_Rangat_A&N_02022022_

In [ ]:
# CELL 4 | Inspect point counts and coordinate types for all villages

print(f"{'Village':<18} {'Points':>15} {'Size MB':>10} {'Coord Type':<14} {'X range'}")
print("-" * 80)

for name, cfg in VILLAGES.items():
    if not os.path.exists(cfg['raw']):
        print(f"  {name:<18} FILE MISSING")
        continue
    size_mb = os.path.getsize(cfg['raw']) / 1e6
    with laspy.open(cfg['raw']) as f:
        total = f.header.point_count
        mins  = f.header.mins
        maxs  = f.header.maxs
    coord_range = max(float(maxs[0]) - float(mins[0]),
                      float(maxs[1]) - float(mins[1]))
    coord_type  = "GEOGRAPHIC" if coord_range < 2 else "PROJECTED"
    VILLAGES[name]['pts']         = total
    VILLAGES[name]['coord_range'] = coord_range
    print(f"  {name:<18} {total:>15,} {size_mb:>10.1f}   {coord_type:<14} "
          f"  {float(mins[0]):.2f} -> {float(maxs[0]):.2f}")

Village                     Points    Size MB Coord Type     X range
--------------------------------------------------------------------------------
  DEVDI                   64,622,538     1874.1   PROJECTED        258417.10 -> 259517.34
  KHAPRETA               163,743,261     1617.1   PROJECTED        311871.00 -> 312679.51
  PIRAYANKUPPAM          157,925,322     4737.8   GEOGRAPHIC       79.86 -> 79.88
  DHUNDA                 172,862,229     1642.6   PROJECTED        633369.95 -> 633994.82
  DHAL                    23,431,282      679.5   PROJECTED        562230.86 -> 562803.17
  THANDALAM              188,077,336     5642.3   GEOGRAPHIC       80.00 -> 80.02
  REFLIGHT_64334          57,635,469     1671.4   PROJECTED        -213.59 -> 764.99
  CHAKHIRASINGH            9,839,175      255.8   PROJECTED        437199.22 -> 438109.44
  GANDHINAGAR_DIG        287,661,850     2036.3   PROJECTED        500229.95 -> 503963.66
  KADAMTALA_RNG        1,650,723,422    11600.7   PROJECTED  

In [ ]:
# CELL 5C | Full pipeline — dynamic scaling, 12 features, morphology, gravity routing, per-village reports, ML validation

import warnings
warnings.filterwarnings('ignore')
from rasterio.transform import from_origin
from rasterio.features   import shapes
from scipy.ndimage import (median_filter, gaussian_filter, uniform_filter,
                            minimum_filter, laplace,
                            label as scipy_label, center_of_mass,
                            binary_opening, binary_closing, generic_filter)
from sklearn.metrics import (confusion_matrix, classification_report,
                              ConfusionMatrixDisplay)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import json, textwrap

def _save_ground_las(name, ground_pts):
    """Write classified ground points to a LAS 1.2 file (class 2)."""
    header              = laspy.LasHeader(point_format=2, version="1.2")
    header.scales       = np.array([0.001, 0.001, 0.001])
    header.offsets      = ground_pts.min(axis=0)
    glas                = laspy.LasData(header)
    glas.x              = ground_pts[:, 0]
    glas.y              = ground_pts[:, 1]
    glas.z              = ground_pts[:, 2]
    glas.classification = np.full(len(ground_pts), 2, dtype=np.uint8)
    glas.write(f'{OUT_BASE}/ground/{name}_ground.las')
    print(f"  Saved LAS      : {OUT_BASE}/ground/{name}_ground.las")
    del glas

# ─────────────────────────────────────────────────────────────────────────
# GROUND CLASSIFICATION — REGULAR
# ─────────────────────────────────────────────────────────────────────────
def classify_ground_regular(name, cfg):
    print(f"\n[{name}] Ground Classification (Regular)")
    las = laspy.read(cfg['raw'])
    x, y, z   = np.array(las.x), np.array(las.y), np.array(las.z)
    total_pts = len(x)
    print(f"  Raw points : {total_pts:,}")
    del las; gc.collect()

    if len(x) > MAX_PTS:
        xi    = ((x - x.min()) / 0.2).astype(np.int32)
        yi    = ((y - y.min()) / 0.2).astype(np.int32)
        keys  = xi.astype(np.int64) * 100_000 + yi.astype(np.int64)
        order = np.argsort(keys)
        keys, x, y, z = keys[order], x[order], y[order], z[order]
        _, first = np.unique(keys, return_index=True)
        x, y, z  = x[first], y[first], z[first]
        if len(x) > MAX_PTS:
            idx = np.random.choice(len(x), MAX_PTS, replace=False)
            x, y, z = x[idx], y[idx], z[idx]
        print(f"  Downsampled: {len(x):,}")

    points = np.vstack((x, y, z)).T.astype(np.float64)
    del x, y, z; gc.collect()

    csf_obj = CSF.CSF()
    csf_obj.params.bSloopSmooth     = True
    csf_obj.params.cloth_resolution = 0.5
    csf_obj.params.rigidness        = 3
    csf_obj.params.time_step        = 0.65
    csf_obj.params.class_threshold  = 0.5
    csf_obj.params.interations      = 500
    csf_obj.setPointCloud(points)
    gnd_idx, off_idx = CSF.VecInt(), CSF.VecInt()
    csf_obj.do_filtering(gnd_idx, off_idx)

    gnd        = np.array(gnd_idx)
    ground_pts = points[gnd]
    print(f"  Ground     : {len(gnd):,} ({100*len(gnd)/len(points):.1f}%)")
    del points, gnd, off_idx; gc.collect()

    _save_ground_las(name, ground_pts)
    VILLAGES[name]['total_pts'] = total_pts
    return ground_pts


# ─────────────────────────────────────────────────────────────────────────
# DYNAMIC MEMORY-SCALING PARAMETER CALCULATOR
# ─────────────────────────────────────────────────────────────────────────
def _compute_dynamic_params(total_pts, xmin, xmax, ymin, ymax, is_geographic):
    """
    Computes VOXEL_RES and STRIDE such that:
      Constraint 1: voxel grid  ≤ 4000 × 4000 = 16M cells  (dict RAM)
      Constraint 2: cell_min occupancy ≤ 33M entries        (heap RAM)
      Constraint 3: stride limits raw throughput for 1B+    (GC pressure)
    """
    MAX_GRID_AXIS    = 4_000
    MAX_DICT_ENTRIES = 33_000_000
    MAX_RAW_INFLIGHT = 330_000_000

    x_range = float(xmax - xmin)
    y_range = float(ymax - ymin)

    if is_geographic:
        lat_mid   = (float(ymin) + float(ymax)) / 2.0
        x_range_m = x_range * 111_000.0 * np.cos(np.radians(lat_mid))
        y_range_m = y_range * 111_000.0
    else:
        x_range_m = x_range
        y_range_m = y_range

    min_voxel_m = max(x_range_m, y_range_m) / MAX_GRID_AXIS
    CANDIDATES_M = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50,
                    0.75, 1.00, 1.50, 2.00, 3.00, 5.00, 10.00]
    voxel_m = next((v for v in CANDIDATES_M if v >= min_voxel_m),
                   max(CANDIDATES_M))

    if is_geographic:
        lat_mid   = (float(ymin) + float(ymax)) / 2.0
        VOXEL_RES = voxel_m / (111_000.0 * np.cos(np.radians(lat_mid)))
    else:
        VOXEL_RES = voxel_m
    grid_cols = int(x_range / VOXEL_RES) + 2
    grid_rows = int(y_range / VOXEL_RES) + 2

    worst_case_cells = grid_cols * grid_rows

    stride_from_dict = max(1, int(np.ceil(worst_case_cells / MAX_DICT_ENTRIES)))
    stride_from_pts  = max(1, int(np.ceil(total_pts / MAX_RAW_INFLIGHT)))
    STRIDE = min(max(stride_from_dict, stride_from_pts), 50)

    info = {
        'voxel_m'       : voxel_m,
        'VOXEL_RES'     : VOXEL_RES,
        'grid_cols'     : grid_cols,
        'grid_rows'     : grid_rows,
        'worst_cells_M' : round(worst_case_cells / 1e6, 2),
        'STRIDE'        : STRIDE,
        'effective_pts' : total_pts // STRIDE,
    }
    return VOXEL_RES, STRIDE, grid_cols, grid_rows, info


# ─────────────────────────────────────────────────────────────────────────
# GROUND CLASSIFICATION — CHUNKED + DYNAMIC SCALING (large files)
# ─────────────────────────────────────────────────────────────────────────
def classify_ground_chunked(name, cfg):
    print(f"\n[{name}] Ground Classification (Chunked + NumPy Grid Accumulator)")

    with laspy.open(cfg['raw']) as f:
        hdr       = f.header
        total_pts = int(hdr.point_count)
        xmin_h    = float(hdr.mins[0]); xmax_h = float(hdr.maxs[0])
        ymin_h    = float(hdr.mins[1]); ymax_h = float(hdr.maxs[1])

    coord_range   = max(xmax_h - xmin_h, ymax_h - ymin_h)
    is_geographic = coord_range < 2.0

    # _compute_dynamic_params is unchanged — still calculates VOXEL_RES + STRIDE
    VOXEL_RES, STRIDE, grid_cols, grid_rows, info = _compute_dynamic_params(
        total_pts, xmin_h, xmax_h, ymin_h, ymax_h, is_geographic
    )

    print(f"  Total pts      : {total_pts:,}")
    print(f"  Coord type     : {'GEOGRAPHIC' if is_geographic else 'PROJECTED'}")
    print(f"  Voxel res      : {info['voxel_m']:.3f} m  ({VOXEL_RES:.8f} native)")
    print(f"  Voxel grid     : {grid_cols} x {grid_rows} "
          f"= {info['worst_cells_M']}M cells")
    print(f"  Stride         : 1/{STRIDE}  "
          f"(effective pts: {info['effective_pts']:,})")

    # ── Pre-allocate NumPy grid arrays ────────────────────────────────────
    # Memory: z=float32 (4B) + x,y=float64 (8B each) = 20B/cell
    # Worst case 4000×4000 = 16M cells → 305 MB   (dict = 4.9 GB for same data)
    grid_mem_mb = (grid_rows * grid_cols * 20) / (1024 ** 2)
    print(f"  Grid RAM       : {grid_mem_mb:.1f} MB  "
          f"({grid_rows}r × {grid_cols}c × 20 B/cell)")

    INF32 = np.float32(np.inf)
    min_z = np.full((grid_rows, grid_cols), INF32,  dtype=np.float32)
    min_x = np.zeros((grid_rows, grid_cols),        dtype=np.float64)
    min_y = np.zeros((grid_rows, grid_cols),        dtype=np.float64)

    # ── Chunked min-z accumulation ────────────────────────────────────────
    processed = 0
    chunk_num = 0

    with laspy.open(cfg['raw']) as f:
        for chunk in f.chunk_iterator(2_000_000):
            n_chunk = len(chunk.x)

            # Stride decimation — keeps deterministic every Nth point
            if STRIDE > 1:
                keep = np.arange(0, n_chunk, STRIDE)
                x = np.asarray(chunk.x)[keep].astype(np.float64)
                y = np.asarray(chunk.y)[keep].astype(np.float64)
                z = np.asarray(chunk.z)[keep].astype(np.float32)
                del keep
            else:
                x = np.asarray(chunk.x, dtype=np.float64)
                y = np.asarray(chunk.y, dtype=np.float64)
                z = np.asarray(chunk.z, dtype=np.float32)

            # Grid cell indices
            xi = np.clip(
                ((x - xmin_h) / VOXEL_RES).astype(np.int32), 0, grid_cols - 1
            )
            yi = np.clip(
                ((y - ymin_h) / VOXEL_RES).astype(np.int32), 0, grid_rows - 1
            )

            # ── Vectorized intra-chunk deduplication ──────────────────────
            # Flat row-major index: cell (yi, xi) → yi*grid_cols + xi
            # np.lexsort sorts by flat key first, then z (ascending)
            # → first occurrence per key after sort = minimum z in this chunk
            flat = yi.astype(np.int64) * grid_cols + xi.astype(np.int64)
            order = np.lexsort((z, flat))    # primary: flat, secondary: z asc
            flat_s = flat[order]
            x_s    = x[order]
            y_s    = y[order]
            z_s    = z[order]

            _, first_idx = np.unique(flat_s, return_index=True)
            flat_u = flat_s[first_idx]   # one entry per unique cell in chunk
            x_u    = x_s[first_idx]
            y_u    = y_s[first_idx]
            z_u    = z_s[first_idx]      # minimum z per cell in this chunk

            del flat_s, x_s, y_s, z_s, order
            # ── end dedup ─────────────────────────────────────────────────

            # Recover (row, col) from flat index
            ri = (flat_u // grid_cols).astype(np.int32)
            ci = (flat_u %  grid_cols).astype(np.int32)
            del flat_u, flat, xi, yi

            # Vectorized grid update — only write where this chunk beats current min
            update_mask = z_u < min_z[ri, ci]
            ri_u = ri[update_mask]
            ci_u = ci[update_mask]

            min_z[ri_u, ci_u] = z_u[update_mask]
            min_x[ri_u, ci_u] = x_u[update_mask]
            min_y[ri_u, ci_u] = y_u[update_mask]

            processed += n_chunk
            chunk_num += 1

            del x, y, z, x_u, y_u, z_u, ri, ci, update_mask, ri_u, ci_u
            gc.collect()   # every chunk — grid arrays stay, temporaries are freed

            if chunk_num % 20 == 0:
                filled = int(np.sum(np.isfinite(min_z)))
                print(f"  Processed {processed:,} / {total_pts:,} pts  "
                      f"| grid cells filled: {filled:,}")

    # ── Extract populated cells into point array ──────────────────────────
    # This is O(grid_rows*grid_cols) numpy boolean mask — no Python-level loop
    valid_mask = np.isfinite(min_z)
    n_valid    = int(valid_mask.sum())
    print(f"  Grid cells     : {n_valid:,} filled / "
          f"{grid_rows * grid_cols:,} total")

    pts_ds = np.column_stack([
        min_x[valid_mask],
        min_y[valid_mask],
        min_z[valid_mask].astype(np.float64)
    ])                              # shape (n_valid, 3), contiguous float64

    # Immediately free the large grid arrays — pts_ds is all we need
    del min_z, min_x, min_y, valid_mask
    gc.collect()

    # Safety cap (should never trigger if _compute_dynamic_params is correct)
    if len(pts_ds) > MAX_PTS:
        idx    = np.random.choice(len(pts_ds), MAX_PTS, replace=False)
        pts_ds = pts_ds[idx]
        del idx; gc.collect()
        print(f"  Safety cap     : {MAX_PTS:,} pts applied")

    print(f"  Post-cap pts   : {len(pts_ds):,}")

    # ── Reproject geographic → projected ──────────────────────────────────
    if is_geographic:
        from pyproj import Transformer
        t = Transformer.from_crs("EPSG:4326", f"EPSG:{cfg['epsg']}",
                                  always_xy=True)
        gx, gy      = t.transform(pts_ds[:, 0], pts_ds[:, 1])
        pts_ds[:, 0] = gx
        pts_ds[:, 1] = gy
        del gx, gy; gc.collect()
        print(f"  Reprojected    : EPSG:{cfg['epsg']}")

    # ── CSF ground classification ─────────────────────────────────────────
    csf_obj = CSF.CSF()
    csf_obj.params.bSloopSmooth     = True
    csf_obj.params.cloth_resolution = 0.5
    csf_obj.params.rigidness        = 3
    csf_obj.params.time_step        = 0.65
    csf_obj.params.class_threshold  = 0.5
    csf_obj.params.interations      = 500
    csf_obj.setPointCloud(pts_ds)

    gnd_idx = CSF.VecInt(); off_idx = CSF.VecInt()
    csf_obj.do_filtering(gnd_idx, off_idx)

    gnd        = np.array(gnd_idx)
    ground_pts = pts_ds[gnd].copy()
    print(f"  Ground pts     : {len(gnd):,} ({100*len(gnd)/len(pts_ds):.1f}%)")
    print(f"  Off-ground     : {len(np.array(off_idx)):,}")

    del pts_ds, gnd, off_idx, gnd_idx, csf_obj; gc.collect()

    _save_ground_las(name, ground_pts)
    VILLAGES[name]['total_pts'] = total_pts
    return ground_pts


# ─────────────────────────────────────────────────────────────────────────
# DTM
# ─────────────────────────────────────────────────────────────────────────
def generate_dtm(name, cfg, ground_pts):
    print(f"\n[{name}] DTM Generation")
    x, y, z    = ground_pts[:,0], ground_pts[:,1], ground_pts[:,2]
    res        = cfg['res']
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    cols = int((xmax - xmin) / res) + 1
    rows = int((ymax - ymin) / res) + 1
    print(f"  Grid       : {cols} x {rows} @ {res}m")

    VILLAGES[name]['area_m2'] = round((xmax-xmin) * (ymax-ymin), 1)

    dtm = np.full((rows, cols), np.nan, dtype=np.float32)
    xi  = np.clip(((x - xmin) / res).astype(np.int32), 0, cols-1)
    yi  = np.clip(((ymax - y) / res).astype(np.int32), 0, rows-1)
    for i in range(len(x)):
        r, c = yi[i], xi[i]
        if np.isnan(dtm[r,c]) or z[i] < dtm[r,c]:
            dtm[r,c] = z[i]

    dtm_f = np.where(np.isnan(dtm), 0.0, dtm).astype(np.float32)
    wmask = (~np.isnan(dtm)).astype(np.float32)
    denom = np.maximum(uniform_filter(wmask, size=5), 1e-6)
    dtm   = (uniform_filter(dtm_f, size=5) / denom).astype(np.float32)
    dtm   = median_filter(dtm, size=3).astype(np.float32)
    dtm   = gaussian_filter(dtm.astype(np.float64), sigma=1).astype(np.float32)
    dtm[~np.isfinite(dtm)] = NODATA

    valid = dtm[dtm != NODATA]
    print(f"  Z range    : {valid.min():.2f}m -> {valid.max():.2f}m")
    del dtm_f, wmask, denom; gc.collect()

    transform = from_origin(xmin, ymax, res, res)
    profile   = {
        'driver': 'GTiff', 'height': rows, 'width': cols,
        'count': 1, 'dtype': 'float32',
        'crs': f'EPSG:{cfg["epsg"]}', 'transform': transform,
        'nodata': NODATA, 'compress': 'lzw',
        'tiled': True, 'blockxsize': 256, 'blockysize': 256, 'predictor': 2
    }
    local = f'/tmp/mopr/{name}_dtm.tif'
    out   = f'{OUT_BASE}/dtm/{name}_dtm_cog.tif'
    with rasterio.open(local, 'w', **profile) as dst:
        dst.write(dtm, 1)
    shutil.copy2(local, out)
    print(f"  OGC COG    : {out}")
    return local, dtm, transform, profile


# ─────────────────────────────────────────────────────────────────────────
# SLOPE + FLOW ACCUMULATION
# ─────────────────────────────────────────────────────────────────────────
def compute_slope(name, dtm_arr, profile):
    import whitebox
    wbt = whitebox.WhiteboxTools(); wbt.verbose = False
    dtm_wbt    = f'/tmp/mopr/{name}_dtm_wbt.tif'
    valid_mean = float(np.nanmean(dtm_arr[dtm_arr != NODATA]))
    dtm_filled = dtm_arr.copy()
    dtm_filled[dtm_filled == NODATA] = valid_mean
    p = profile.copy(); p['nodata'] = None
    with rasterio.open(dtm_wbt, 'w', **p) as dst:
        dst.write(dtm_filled.astype(np.float32), 1)
    del dtm_filled

    slope_path = f'/tmp/mopr/{name}_slope.tif'
    try:
        wbt.slope(dtm_wbt, slope_path, zfactor=1.0)
        with rasterio.open(slope_path) as src:
            slope = src.read(1).astype(np.float32)
            nd    = src.nodata
            if nd is not None: slope[slope == nd] = np.nan
        os.remove(slope_path)
    except Exception as e:
        print(f"  WBT slope fallback ({e})")
        dy, dx = np.gradient(np.nan_to_num(dtm_arr.copy(), nan=0))
        slope  = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2))).astype(np.float32)
    os.remove(dtm_wbt)
    return slope


def numpy_flow_accumulation(dtm_arr):
    dtm_c = dtm_arr.copy().astype(np.float32)
    dtm_c[dtm_c == NODATA] = np.nan
    dtm_f = np.nan_to_num(dtm_c, nan=float(np.nanmax(dtm_c)))

    r5  = dtm_f - minimum_filter(dtm_f, size=5)
    r21 = dtm_f - minimum_filter(dtm_f, size=21)
    r51 = dtm_f - minimum_filter(dtm_f, size=51)

    accum = (0.2*np.exp(-r5  / (r5.std()  + 1e-6))
           + 0.3*np.exp(-r21 / (r21.std() + 1e-6))
           + 0.5*np.exp(-r51 / (r51.std() + 1e-6))).astype(np.float32)
    accum = gaussian_filter(accum, sigma=3).astype(np.float32)
    accum[dtm_arr == NODATA] = 0.0
    del r5, r21, r51, dtm_f; gc.collect()
    return accum


# ─────────────────────────────────────────────────────────────────────────
# 12-FEATURE MATRIX WITH NEIGHBORHOOD STATISTICS
# ─────────────────────────────────────────────────────────────────────────
def build_feature_matrix(dtm_arr, slope, accum, twi):
    """
    12 features per pixel:
      Point      : elevation, slope, accum, twi, curvature, dist_outlet
      5x5 window : mean_elev, std_elev, mean_slope, mean_twi
      11x11 window: mean_accum, range_elev
    """
    dtm_c = dtm_arr.copy().astype(np.float32)
    dtm_c[dtm_c == NODATA] = np.nan
    dtm_n = np.nan_to_num(dtm_c, nan=0)
    sl_n  = np.nan_to_num(slope,  nan=0)
    ac_n  = np.nan_to_num(accum,  nan=0)
    tw_n  = np.nan_to_num(twi,    nan=0)

    curv = laplace(dtm_n).astype(np.float32)
    fi   = np.argmin(np.nan_to_num(dtm_c, nan=9999))
    rc, cc = np.unravel_index(fi, dtm_c.shape)
    dist = np.sqrt(
        (np.arange(dtm_c.shape[0]).reshape(-1,1) - rc)**2 +
        (np.arange(dtm_c.shape[1]).reshape(1,-1) - cc)**2
    ).astype(np.float32)

    mean_elev_5  = uniform_filter(dtm_n, size=5).astype(np.float32)
    mean_slope_5 = uniform_filter(sl_n,  size=5).astype(np.float32)
    mean_twi_5   = uniform_filter(tw_n,  size=5).astype(np.float32)
    std_elev_5   = np.sqrt(np.maximum(
        uniform_filter(dtm_n**2, size=5) - mean_elev_5**2, 0
    )).astype(np.float32)

    mean_accum_11 = uniform_filter(ac_n,  size=11).astype(np.float32)
    min_elev_11   = minimum_filter(dtm_n, size=11).astype(np.float32)
    range_elev_11 = (mean_elev_5 - min_elev_11).astype(np.float32)

    feats = np.stack([
        dtm_n, sl_n, ac_n, tw_n, curv, dist,
        mean_elev_5, std_elev_5,
        mean_slope_5, mean_twi_5,
        mean_accum_11, range_elev_11
    ], axis=-1)

    del dtm_n, sl_n, ac_n, tw_n, curv, dist
    del mean_elev_5, std_elev_5, mean_slope_5, mean_twi_5
    del mean_accum_11, range_elev_11, min_elev_11
    gc.collect()
    return feats


# ─────────────────────────────────────────────────────────────────────────
# XGBOOST TRAIN / PREDICT
# ─────────────────────────────────────────────────────────────────────────
def train_xgboost(feats, rule_hotspot):
    rows, cols, nf = feats.shape
    X = feats.reshape(-1, nf)
    y = rule_hotspot.flatten()
    MAX_TRAIN = 200_000
    if len(X) > MAX_TRAIN:
        idx = np.random.choice(len(X), MAX_TRAIN, replace=False)
        X, y = X[idx], y[idx]
    scaler = StandardScaler()
    Xs     = scaler.fit_transform(X)
    model  = xgb.XGBClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        subsample=0.8, eval_metric='mlogloss',
        n_jobs=-1, verbosity=0, scale_pos_weight=3
    )
    model.fit(Xs, y)
    vals, cnts = np.unique(y, return_counts=True)
    print(f"  Train labels   : { {int(v):int(c) for v,c in zip(vals,cnts)} }")
    print(f"  XGBoost trained: {len(X):,} px | {nf} features")
    return model, scaler


def predict_ml(feats, model, scaler):
    rows, cols, nf = feats.shape
    X     = feats.reshape(-1, nf)
    preds = np.zeros(len(X), dtype=np.uint8)
    conf  = np.zeros(len(X), dtype=np.float32)
    CHUNK = 500_000
    for s in range(0, len(X), CHUNK):
        e         = min(s+CHUNK, len(X))
        Xc        = scaler.transform(X[s:e])
        p         = model.predict_proba(Xc)
        preds[s:e] = model.predict(Xc).astype(np.uint8)
        conf[s:e]  = p[:,2] if p.shape[1] > 2 else p[:,1]
    return preds.reshape(rows,cols), conf.reshape(rows,cols)


# ─────────────────────────────────────────────────────────────────────────
# MORPHOLOGICAL POST-PROCESSING
# ─────────────────────────────────────────────────────────────────────────
def morphological_cleanup(hotspot_ml, dtm_arr):
    """
    binary_opening (3x3): removes isolated salt-and-pepper noise pixels.
    binary_closing (5x5): fills small holes within hotspot clusters.
    Applied per class (High then Medium) to preserve class labels.
    """
    struct_open  = np.ones((3,3), dtype=bool)
    struct_close = np.ones((5,5), dtype=bool)
    outside      = (dtm_arr == NODATA)
    cleaned      = np.zeros_like(hotspot_ml, dtype=np.uint8)

    for cls in [2, 1]:
        mask = (hotspot_ml == cls)
        mask = binary_opening(mask,  structure=struct_open)
        mask = binary_closing(mask,  structure=struct_close)
        mask[outside] = False
        cleaned[mask] = cls

    removed = int((hotspot_ml > 0).sum()) - int((cleaned > 0).sum())
    print(f"  Morphology: removed {removed:,} noise pixels")
    print(f"  Final High : {(cleaned==2).sum():,} | Medium: {(cleaned==1).sum():,}")
    return cleaned


# ─────────────────────────────────────────────────────────────────────────
# RISK → RGBA
# ─────────────────────────────────────────────────────────────────────────
def risk_to_rgba(risk_arr, dtm_arr):
    """
    Convert risk raster to RGBA image with correct transparency.

    NODATA (-9999) = outside village boundary       → fully transparent
    0              = safe/dry ground inside village → light green
    1              = medium waterlogging risk       → orange
    2              = high waterlogging risk         → red
    """
    outside = (risk_arr < -100) | (dtm_arr == NODATA)
    cls     = np.clip(np.round(risk_arr), 0, 2).astype(np.uint8)
    cls[outside] = 255   # sentinel for transparent

    COLOR = {
        255: (0.00, 0.00, 0.00, 0.00),   # transparent — outside village
          0: (0.82, 0.94, 0.78, 1.00),   # light green — safe/dry ground
          1: (1.00, 0.55, 0.00, 1.00),   # orange      — medium risk
          2: (0.80, 0.00, 0.00, 1.00),   # red         — high risk
    }
    rgba = np.zeros((*cls.shape, 4), dtype=np.float32)
    for val, color in COLOR.items():
        rgba[cls == val] = color

    return rgba, cls


# ─────────────────────────────────────────────────────────────────────────
# GRAVITY-CORRECT DIJKSTRA DRAINAGE ROUTING
# ─────────────────────────────────────────────────────────────────────────
def design_drainage_network(name, hotspot_final, stream_raster,
                             dtm_arr, accum, transform, crs):
    """
    cost = (1 - norm_accum)*60 + norm_elev*40 + 1

    Low cost along natural valleys (high accum + low elevation) ensures
    all proposed drains are gravity-fed and follow existing drainage axes.
    NODATA cells = 9999 (impenetrable) — drains never exit village boundary.
    """
    dtm_c = dtm_arr.copy().astype(np.float32)
    dtm_c[dtm_c == NODATA] = np.nan
    dtm_f = np.nan_to_num(dtm_c, nan=float(np.nanmax(dtm_c)))

    def norm(a):
        mn, mx = a.min(), a.max()
        return (a - mn) / (mx - mn + 1e-9)

    cost = ((1 - norm(accum)) * 60 + norm(dtm_f) * 40 + 1).astype(np.float32)
    cost[dtm_arr == NODATA] = 9999.0

    labeled, n_clust = scipy_label((hotspot_final == 2).astype(np.uint8))
    print(f"  Hotspot clusters : {n_clust}")

    stream_cells = np.argwhere(stream_raster > 0)
    if len(stream_cells) == 0:
        print("  No stream cells — skipping drain routing")
        return gpd.GeoDataFrame(geometry=[], crs=crs)

    targets = set(map(tuple, stream_cells.tolist()))

    def px_to_coord(r, c):
        return (transform.c + c*transform.a + 0.5*transform.a,
                transform.f + r*transform.e + 0.5*transform.e)

    def dijkstra(cost_arr, start, targets):
        rows, cols = cost_arr.shape
        dist  = np.full((rows,cols), np.inf, dtype=np.float32)
        prev  = {}
        dist[start] = 0
        heap = [(0.0, start)]
        dirs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
        while heap:
            d, u = heapq.heappop(heap)
            if u in targets:
                path = []
                while u in prev: path.append(u); u = prev[u]
                path.append(start)
                return list(reversed(path))
            if d > dist[u]: continue
            r, c = u
            for dr, dc in dirs:
                nr, nc = r+dr, c+dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    nd = d + cost_arr[nr,nc]*(1.414 if dr and dc else 1.0)
                    if nd < dist[nr,nc]:
                        dist[nr,nc] = nd
                        prev[(nr,nc)] = u
                        heapq.heappush(heap, (nd,(nr,nc)))
        return []

    drain_lines  = []
    total_length = 0.0
    for cid in range(1, min(n_clust+1, 51)):
        mask = (labeled == cid)
        if mask.sum() < 25: continue
        cy, cx = center_of_mass(mask)
        path   = dijkstra(cost, (int(cy),int(cx)), targets)
        if len(path) < 2: continue
        coords = [px_to_coord(r,c) for r,c in path]
        line   = LineString(coords)
        total_length += line.length
        drain_lines.append({'geometry': line, 'cluster_id': int(cid),
                            'hotspot_area': int(mask.sum()),
                            'length_m': round(line.length,2),
                            'village': name, 'type': 'proposed_drain'})

    drains_gdf = gpd.GeoDataFrame(drain_lines, crs=crs)
    print(f"  Proposed drains  : {len(drains_gdf)} | "
          f"Total length: {total_length/1000:.2f} km")
    VILLAGES[name]['drain_length_km'] = round(total_length/1000, 2)
    return drains_gdf


# ─────────────────────────────────────────────────────────────────────────
# SAVE COG HELPER
# ─────────────────────────────────────────────────────────────────────────
def save_cog(name, arr, tag, profile, dtype='float32'):
    p = profile.copy()
    p.update({'dtype': dtype,
              'nodata': 0 if dtype == 'uint8' else NODATA})
    with rasterio.open(f'{OUT_BASE}/rasters/{name}_{tag}_cog.tif',
                       'w', **p) as dst:
        dst.write(arr.astype(dtype), 1)


# ─────────────────────────────────────────────────────────────────────────
# PER-VILLAGE REPORT WITH ML VALIDATION
# ─────────────────────────────────────────────────────────────────────────
def generate_village_report(name, cfg, dtm_arr, slope, accum, twi,
                             rule_hot, hotspot_final, drains_gdf,
                             streams_gdf, hotspots_gdf, transform):
    fig = plt.figure(figsize=(20, 14))
    fig.patch.set_facecolor('#1a1a2e')
    title_color = 'white'
    gs = fig.add_gridspec(3, 4, hspace=0.4, wspace=0.35)

    def clean(arr, nd=NODATA):
        a = arr.copy().astype(np.float32)
        a[a == nd] = np.nan
        return a

    # ── Panel 1: DTM ──────────────────────────────────────────────────────
    ax1   = fig.add_subplot(gs[0, :2])
    dtm_v = clean(dtm_arr)
    valid = dtm_v[~np.isnan(dtm_v)]
    im1   = ax1.imshow(dtm_v, cmap='terrain', interpolation='bilinear',
                       vmin=np.percentile(valid,1), vmax=np.percentile(valid,99))
    plt.colorbar(im1, ax=ax1, label='Elevation (m)', shrink=0.8)
    ax1.set_title(f'{name} — Digital Terrain Model',
                  color=title_color, fontweight='bold')
    ax1.axis('off')

    # ── Panel 2: Risk + Drains (4-class RGBA, transparent NODATA) ────────
    ax2 = fig.add_subplot(gs[0, 2:])
    ax2.set_facecolor('white')
    rgba, cls = risk_to_rgba(hotspot_final.astype(np.float32), dtm_arr)
    ax2.imshow(rgba, interpolation='nearest')

    if len(drains_gdf) > 0:
        xmin_t = transform.c; ymax_t = transform.f
        res_x  = transform.a; res_y  = abs(transform.e)
        for _, row_d in drains_gdf.iterrows():
            coords = list(row_d.geometry.coords)
            px_c   = [(c[0] - xmin_t) / res_x for c in coords]
            px_r   = [(ymax_t - c[1]) / res_y  for c in coords]
            ax2.plot(px_c, px_r, color='cyan', linewidth=1.5,
                     linestyle='--', alpha=0.9, zorder=5)

    ax2.legend(handles=[
        mpatches.Patch(color='#D1F0C7', label='Safe / Dry Ground'),
        mpatches.Patch(color='#FF8C00', label='Medium Risk'),
        mpatches.Patch(color='#CC0000', label='High Risk'),
        Line2D([0],[0], color='cyan', linestyle='--',
               linewidth=2, label='Proposed Drain'),
    ], loc='lower right', fontsize=8, facecolor='#333333', labelcolor='white')
    ax2.set_title(f'{name} — Waterlogging Risk + Proposed Drains',
                  color=title_color, fontweight='bold')
    ax2.axis('off')

    # ── Panel 3: Confusion Matrix ─────────────────────────────────────────
    ax3    = fig.add_subplot(gs[1, :2])
    y_true = rule_hot.flatten()
    y_pred = hotspot_final.flatten()
    MAX_CM = 500_000
    if len(y_true) > MAX_CM:
        idx    = np.random.choice(len(y_true), MAX_CM, replace=False)
        y_true = y_true[idx]; y_pred = y_pred[idx]

    cm      = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im3     = ax3.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax3.set_xticks([0,1,2]); ax3.set_yticks([0,1,2])
    ax3.set_xticklabels(['No Risk','Medium','High'], color=title_color)
    ax3.set_yticklabels(['No Risk','Medium','High'], color=title_color)
    ax3.set_xlabel('ML Predicted',     color=title_color)
    ax3.set_ylabel('Rule-Based True',  color=title_color)
    ax3.set_title('Confusion Matrix (ML vs Rule)',
                  color=title_color, fontweight='bold')
    for i in range(3):
        for j in range(3):
            ax3.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center',
                     va='center', color='black', fontsize=10)
    plt.colorbar(im3, ax=ax3, shrink=0.8)
    ax3.tick_params(colors=title_color)

    # ── Panel 4: Classification Report Table ─────────────────────────────
    ax4 = fig.add_subplot(gs[1, 2:])
    ax4.axis('off')
    report = classification_report(
        y_true, y_pred, labels=[0,1,2],
        target_names=['No Risk','Medium','High'],
        output_dict=True, zero_division=0)

    rows_r  = ['No Risk','Medium','High','accuracy','macro avg','weighted avg']
    table_d = []
    for r in rows_r:
        if r not in report: continue
        rd = report[r]
        if isinstance(rd, dict):
            table_d.append([r,
                f"{rd.get('precision',0):.3f}",
                f"{rd.get('recall',0):.3f}",
                f"{rd.get('f1-score',0):.3f}",
                f"{int(rd.get('support',0)):,}"])
        else:
            table_d.append([r, '', '', f"{rd:.3f}", ''])

    tbl = ax4.table(cellText=table_d,
                    colLabels=['Class','Precision','Recall','F1','Support'],
                    loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
    for (row_i, col_i), cell in tbl.get_celld().items():
        cell.set_facecolor('#2d2d44' if row_i % 2 == 0 else '#1a1a2e')
        cell.set_text_props(color='white')
        cell.set_edgecolor('#555555')
    ax4.set_title('ML Classification Report',
                  color=title_color, fontweight='bold')

    # ── Panel 5: Stats Summary Bar ────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, :])
    ax5.axis('off')
    ax5.set_facecolor('#1a1a2e')
    n_high    = int((hotspot_final==2).sum())
    n_med     = int((hotspot_final==1).sum())
    area_km2  = round(VILLAGES[name].get('area_m2',  0) / 1e6, 3)
    total_pts = VILLAGES[name].get('total_pts', 0)
    drain_km  = VILLAGES[name].get('drain_length_km', 0.0)

    summary_text = (
        f"Village: {name}   |   Total LiDAR Points: {total_pts:,}   |   "
        f"Survey Area: {area_km2} km²   |   High Risk Cells: {n_high:,}   |   "
        f"Medium Risk Cells: {n_med:,}   |   "
        f"Proposed Drain Length: {drain_km} km   |   "
        f"Proposed Drains: {len(drains_gdf)}"
    )
    ax5.text(0.5, 0.5, summary_text, transform=ax5.transAxes,
             ha='center', va='center', fontsize=10, color='white',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#2d2d44',
                       edgecolor='#4488ff'))

    fig.suptitle(f'GramDrain — Village Analysis Report: {name}',
                 fontsize=16, fontweight='bold', color='white', y=0.98)
    out = f'{OUT_BASE}/village_reports/{name}_report.png'
    plt.savefig(out, dpi=120, bbox_inches='tight', facecolor='#1a1a2e')
    plt.close(fig)
    print(f"  Report saved : {out}")
    return report


# ─────────────────────────────────────────────────────────────────────────
# FULL VILLAGE PROCESSOR
# ─────────────────────────────────────────────────────────────────────────
def process_village(name, cfg, ground_pts,
                    xgb_model=None, xgb_scaler=None):

    # 1. DTM
    dtm_entry = generate_dtm(name, cfg, ground_pts)
    dtm_path, dtm_arr, transform, profile = dtm_entry
    del ground_pts; gc.collect()

    # 2. Slope + Flow Accumulation
    print(f"\n[{name}] Terrain Derivatives")
    slope = compute_slope(name, dtm_arr, profile)
    accum = numpy_flow_accumulation(dtm_arr)

    stream_thresh = float(np.percentile(accum[accum > 0], 97))
    stream_raster = (accum >= stream_thresh).astype(np.uint8)
    stream_raster[dtm_arr == NODATA] = 0
    print(f"  Slope range  : {np.nanmin(slope):.2f} -> {np.nanmax(slope):.2f} deg")
    print(f"  Stream cells : {stream_raster.sum():,}")

    # 3. TWI
    dtm_c     = dtm_arr.copy().astype(np.float32)
    dtm_c[dtm_c == NODATA] = np.nan
    slope_rad = np.deg2rad(np.clip(np.nan_to_num(slope), 0.01, 89))
    twi       = np.log((accum*100 + 1) / (np.tan(slope_rad) + 1e-6)).astype(np.float32)
    twi       = np.nan_to_num(twi)
    gc.collect()

    # 4. Rule-based labels (training targets for XGBoost)
    def norm(a):
        a = np.nan_to_num(a).astype(np.float32)
        return (a - a.min()) / (a.max() - a.min() + 1e-9)

    risk_rule = norm(0.40*norm(twi) + 0.35*norm(accum)
                   + 0.15*(1-norm(slope)) + 0.10*(1-norm(dtm_c)))
    rule_hot  = np.zeros_like(risk_rule, dtype=np.uint8)
    rule_hot[risk_rule > 0.92]                          = 2
    rule_hot[(risk_rule >= 0.78) & (risk_rule <= 0.92)] = 1
    rule_hot[dtm_arr == NODATA] = 0
    print(f"\n[{name}] Rule labels  — High: {(rule_hot==2).sum():,} "
          f"| Medium: {(rule_hot==1).sum():,}")

    # 5. XGBoost (per-village retrain with 12-feature matrix)
    print(f"\n[{name}] XGBoost")
    feats = build_feature_matrix(dtm_arr, slope, accum, twi)
    xgb_model, xgb_scaler = train_xgboost(feats, rule_hot)
    hotspot_ml, confidence = predict_ml(feats, xgb_model, xgb_scaler)
    print(f"  Raw ML — High: {(hotspot_ml==2).sum():,} "
          f"| Medium: {(hotspot_ml==1).sum():,}")
    del feats; gc.collect()

    # 6. Morphological cleanup
    print(f"\n[{name}] Morphological Post-processing")
    hotspot_ml = morphological_cleanup(hotspot_ml, dtm_arr)

    # 7. Fallback to rule-based if ML completely underperforms
    ml_count = int((hotspot_ml==2).sum()) + int((hotspot_ml==1).sum())
    if ml_count < 10:
        print(f"  ML underperformed — using rule_hot fallback")
        hotspot_final = rule_hot
    else:
        hotspot_final = hotspot_ml

    # 8. Save rasters (risk has NODATA=-9999 outside village)
    risk_export = hotspot_final.copy().astype(np.float32)
    risk_export[dtm_arr == NODATA] = NODATA
    save_cog(name, risk_export,  'risk',       profile)
    save_cog(name, confidence,   'confidence', profile)
    save_cog(name, slope,        'slope',      profile)
    save_cog(name, twi,          'twi',        profile)
    save_cog(name, accum,        'accum',      profile)
    del risk_export; gc.collect()

    # 9. Drainage network (gravity-correct Dijkstra)
    print(f"\n[{name}] Drainage Network Design")
    crs        = f'EPSG:{cfg["epsg"]}'
    drains_gdf = design_drainage_network(
        name, hotspot_final, stream_raster,
        dtm_arr, accum, transform, crs)

    # 10. Vectorize hotspots + streams
    records = []
    for code, label in {1:'Medium', 2:'High'}.items():
        mask = (hotspot_final == code).astype(np.uint8)
        for geom, val in shapes(mask, transform=transform):
            if val == 1:
                g = shape(geom)
                records.append({'geometry': g, 'risk_level': label,
                                'risk_code': code,
                                'area_m2': round(g.area,2),
                                'village': name})
    hotspots_gdf = gpd.GeoDataFrame(records, crs=crs)

    slope_clean = np.nan_to_num(slope, nan=0)
    stream_phys = ((stream_raster > 0) & (slope_clean > 0.01)).astype(np.uint8)
    stream_geoms = [shape(g) for g,v
                    in shapes(stream_phys, transform=transform) if v == 1]
    streams_gdf = gpd.GeoDataFrame({
        'geometry':      stream_geoms,
        'feature_type':  'drainage_channel',
        'physics_valid': True,
        'village':       name
    }, crs=crs)

    # 11. Export OGC GeoPackage
    gpkg = f'{OUT_BASE}/gpkg/{name}_drainage.gpkg'
    streams_gdf.to_file(gpkg,  layer='streams',  driver='GPKG')
    hotspots_gdf.to_file(gpkg, layer='hotspots', driver='GPKG')
    if len(drains_gdf) > 0:
        drains_gdf.to_file(gpkg, layer='proposed_drains', driver='GPKG')

    # 12. Per-village PNG report
    print(f"\n[{name}] Generating Village Report")
    generate_village_report(
        name, cfg, dtm_arr, slope, accum, twi,
        rule_hot, hotspot_final, drains_gdf,
        streams_gdf, hotspots_gdf, transform)

    print(f"\n[{name}] ── Output Summary ──────────────────────")
    print(f"  Streams         : {len(streams_gdf):,}")
    print(f"  Hotspots        : {len(hotspots_gdf):,}")
    print(f"  Proposed drains : {len(drains_gdf)}")
    print(f"  GPKG            : {gpkg}")

    del rule_hot, slope, accum, twi, dtm_c, slope_rad; gc.collect()

    return (dtm_entry,
            (slope_clean, confidence, hotspot_final, stream_raster),
            (streams_gdf, hotspots_gdf, drains_gdf),
            (xgb_model, xgb_scaler))


print("All pipeline functions defined")

All pipeline functions defined


In [ ]:
# CELL 6 | Process DEVDI | Trains XGBoost on this village

gc.collect()
devdi_ground = classify_ground_regular('DEVDI', VILLAGES['DEVDI'])
devdi_dtm, devdi_deriv, devdi_vectors, (xgb_model, xgb_scaler) = \
    process_village('DEVDI', VILLAGES['DEVDI'], devdi_ground)
print("DEVDI complete")


[DEVDI] Ground Classification (Regular)
  Raw points : 64,622,538
  Downsampled: 5,000,000
  Ground     : 4,185,795 (83.7%)
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/DEVDI_ground.las

[DEVDI] DTM Generation
  Grid       : 1101 x 903 @ 1.0m
  Z range    : -30.18m -> 0.00m
  OGC COG    : /content/drive/MyDrive/mopr_outputs/dtm/DEVDI_dtm_cog.tif

[DEVDI] Terrain Derivatives
Decompressing WhiteboxTools_linux_musl.zip ...
WhiteboxTools package directory: /usr/local/lib/python3.12/dist-packages/whitebox
  Slope range  : 0.00 -> 80.56 deg
  Stream cells : 222,887

[DEVDI] Rule labels  — High: 643,779 | Medium: 201,284

[DEVDI] XGBoost
  Train labels   : {0: 30210, 1: 40193, 2: 129597}
  XGBoost trained: 200,000 px | 12 features
  Raw ML — High: 643,812 | Medium: 201,214

[DEVDI] Morphological Post-processing
  Morphology: removed 4,765 noise pixels
  Final High : 636,109 | Medium: 204,152

[DEVDI] Drainage Network Design
  Hotspot clusters : 57


  Proposed drains  : 42 | Total length: 11.61 km

[DEVDI] Generating Village Report
  Report saved : /content/drive/MyDrive/mopr_outputs/village_reports/DEVDI_report.png

[DEVDI] ── Output Summary ──────────────────────
  Streams         : 0
  Hotspots        : 190
  Proposed drains : 42
  GPKG            : /content/drive/MyDrive/mopr_outputs/gpkg/DEVDI_drainage.gpkg
DEVDI complete


In [ ]:
# CELL 6B | Process REFLIGHT_64334 (point count unknown — auto-detect)

gc.collect()
with laspy.open(VILLAGES['REFLIGHT_64334']['raw']) as f:
    pts = f.header.point_count
print(f"REFLIGHT_64334: {pts:,} points")

if pts > 50_000_000:
    ground = classify_ground_chunked('REFLIGHT_64334', VILLAGES['REFLIGHT_64334'])
else:
    ground = classify_ground_regular('REFLIGHT_64334', VILLAGES['REFLIGHT_64334'])

ref_dtm, ref_deriv, ref_vectors, _ = \
    process_village('REFLIGHT_64334', VILLAGES['REFLIGHT_64334'], ground)
print("REFLIGHT_64334 complete")

REFLIGHT_64334: 57,635,469 points

[REFLIGHT_64334] Ground Classification (Chunked + NumPy Grid Accumulator)
  Total pts      : 57,635,469
  Coord type     : PROJECTED
  Voxel res      : 0.300 m  (0.30000000 native)
  Voxel grid     : 3263 x 3043 = 9.93M cells
  Stride         : 1/1  (effective pts: 57,635,469)
  Grid RAM       : 189.4 MB  (3043r × 3263c × 20 B/cell)
  Processed 40,000,000 / 57,635,469 pts  | grid cells filled: 5,760,689
  Grid cells     : 8,277,471 filled / 9,929,309 total
  Safety cap     : 5,000,000 pts applied
  Post-cap pts   : 5,000,000
  Ground pts     : 4,498,962 (90.0%)
  Off-ground     : 501,038
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/REFLIGHT_64334_ground.las

[REFLIGHT_64334] DTM Generation
  Grid       : 979 x 913 @ 1.0m
  Z range    : 0.00m -> 132.59m
  OGC COG    : /content/drive/MyDrive/mopr_outputs/dtm/REFLIGHT_64334_dtm_cog.tif

[REFLIGHT_64334] Terrain Derivatives
  Slope range  : 0.00 -> 89.17 deg
  Stream cells : 85,817

[REFL

In [ ]:
# CELL 7 | Process KHAPRETA | Chunked loader | Uses trained XGBoost model

gc.collect()
khap_ground = classify_ground_chunked('KHAPRETA', VILLAGES['KHAPRETA'])
khap_dtm, khap_deriv, khap_vectors, _ = \
    process_village('KHAPRETA', VILLAGES['KHAPRETA'],
                    khap_ground, xgb_model, xgb_scaler)
print("KHAPRETA complete")


[KHAPRETA] Ground Classification (Chunked + NumPy Grid Accumulator)
  Total pts      : 163,743,261
  Coord type     : PROJECTED
  Voxel res      : 0.300 m  (0.30000000 native)
  Voxel grid     : 2697 x 2754 = 7.43M cells
  Stride         : 1/1  (effective pts: 163,743,261)
  Grid RAM       : 141.7 MB  (2754r × 2697c × 20 B/cell)
  Processed 40,000,000 / 163,743,261 pts  | grid cells filled: 1,230,437
  Processed 80,000,000 / 163,743,261 pts  | grid cells filled: 2,325,252
  Processed 120,000,000 / 163,743,261 pts  | grid cells filled: 3,423,269
  Processed 160,000,000 / 163,743,261 pts  | grid cells filled: 4,618,598
  Grid cells     : 4,726,460 filled / 7,427,538 total
  Post-cap pts   : 4,726,460
  Ground pts     : 4,325,573 (91.5%)
  Off-ground     : 400,887
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/KHAPRETA_ground.las

[KHAPRETA] DTM Generation
  Grid       : 809 x 826 @ 1.0m
  Z range    : 0.00m -> 148.34m
  OGC COG    : /content/drive/MyDrive/mopr_outputs/dtm

In [ ]:
# CELL 7B | Process CHAKHIRASINGH (auto-detect)

gc.collect()
with laspy.open(VILLAGES['CHAKHIRASINGH']['raw']) as f:
    pts = f.header.point_count
print(f"CHAKHIRASINGH: {pts:,} points")

if pts > 50_000_000:
    ground = classify_ground_chunked('CHAKHIRASINGH', VILLAGES['CHAKHIRASINGH'])
else:
    ground = classify_ground_regular('CHAKHIRASINGH', VILLAGES['CHAKHIRASINGH'])

chak_dtm, chak_deriv, chak_vectors, _ = \
    process_village('CHAKHIRASINGH', VILLAGES['CHAKHIRASINGH'], ground)
print("CHAKHIRASINGH complete")

CHAKHIRASINGH: 9,839,175 points

[CHAKHIRASINGH] Ground Classification (Regular)
  Raw points : 9,839,175
  Downsampled: 5,000,000
  Ground     : 3,716,341 (74.3%)
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/CHAKHIRASINGH_ground.las

[CHAKHIRASINGH] DTM Generation
  Grid       : 910 x 997 @ 1.0m
  Z range    : 0.00m -> 146.88m
  OGC COG    : /content/drive/MyDrive/mopr_outputs/dtm/CHAKHIRASINGH_dtm_cog.tif

[CHAKHIRASINGH] Terrain Derivatives
  Slope range  : 0.00 -> 89.30 deg
  Stream cells : 216,631

[CHAKHIRASINGH] Rule labels  — High: 247,146 | Medium: 191,482

[CHAKHIRASINGH] XGBoost
  Train labels   : {0: 103115, 1: 42442, 2: 54443}
  XGBoost trained: 200,000 px | 12 features
  Raw ML — High: 247,157 | Medium: 191,407

[CHAKHIRASINGH] Morphological Post-processing
  Morphology: removed 10,654 noise pixels
  Final High : 240,653 | Medium: 187,257

[CHAKHIRASINGH] Drainage Network Design
  Hotspot clusters : 7
  Proposed drains  : 3 | Total length: 0.71 km

[CHAKH

In [ ]:
# CELL 8 | Process PIRAYANKUPPAM | Chunked loader | Uses trained XGBoost model

gc.collect()
pira_ground = classify_ground_chunked('PIRAYANKUPPAM', VILLAGES['PIRAYANKUPPAM'])
pira_dtm, pira_deriv, pira_vectors, _ = \
    process_village('PIRAYANKUPPAM', VILLAGES['PIRAYANKUPPAM'],
                    pira_ground, xgb_model, xgb_scaler)
print("PIRAYANKUPPAM complete")


[PIRAYANKUPPAM] Ground Classification (Chunked + NumPy Grid Accumulator)
  Total pts      : 157,925,322
  Coord type     : GEOGRAPHIC
  Voxel res      : 0.500 m  (0.00000463 native)
  Voxel grid     : 3049 x 2857 = 8.71M cells
  Stride         : 1/1  (effective pts: 157,925,322)
  Grid RAM       : 166.1 MB  (2857r × 3049c × 20 B/cell)
  Processed 40,000,000 / 157,925,322 pts  | grid cells filled: 1,411,918
  Processed 80,000,000 / 157,925,322 pts  | grid cells filled: 3,115,590
  Processed 120,000,000 / 157,925,322 pts  | grid cells filled: 4,704,070
  Grid cells     : 6,491,969 filled / 8,710,993 total
  Safety cap     : 5,000,000 pts applied
  Post-cap pts   : 5,000,000
  Reprojected    : EPSG:32644
  Ground pts     : 4,568,041 (91.4%)
  Off-ground     : 431,959
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/PIRAYANKUPPAM_ground.las

[PIRAYANKUPPAM] DTM Generation
  Grid       : 1528 x 1465 @ 1.0m
  Z range    : -2.05m -> 61.69m
  OGC COG    : /content/drive/MyDrive/m

In [ ]:
# CELL 8B | Process GANDHINAGAR_DIG — Andaman, UTM 46N (auto-detect size)

gc.collect()
with laspy.open(VILLAGES['GANDHINAGAR_DIG']['raw']) as f:
    pts = f.header.point_count
print(f"GANDHINAGAR_DIG: {pts:,} points")

if pts > 50_000_000:
    ground = classify_ground_chunked('GANDHINAGAR_DIG', VILLAGES['GANDHINAGAR_DIG'])
else:
    ground = classify_ground_regular('GANDHINAGAR_DIG', VILLAGES['GANDHINAGAR_DIG'])

gand_dtm, gand_deriv, gand_vectors, _ = \
    process_village('GANDHINAGAR_DIG', VILLAGES['GANDHINAGAR_DIG'], ground)
print("GANDHINAGAR_DIG complete")

GANDHINAGAR_DIG: 287,661,850 points

[GANDHINAGAR_DIG] Ground Classification (Chunked + NumPy Grid Accumulator)
  Total pts      : 287,661,850
  Coord type     : PROJECTED
  Voxel res      : 1.000 m  (1.00000000 native)
  Voxel grid     : 3735 x 2859 = 10.68M cells
  Stride         : 1/1  (effective pts: 287,661,850)
  Grid RAM       : 203.7 MB  (2859r × 3735c × 20 B/cell)
  Processed 40,000,000 / 287,661,850 pts  | grid cells filled: 705,999
  Processed 80,000,000 / 287,661,850 pts  | grid cells filled: 1,378,436
  Processed 120,000,000 / 287,661,850 pts  | grid cells filled: 2,099,650
  Processed 160,000,000 / 287,661,850 pts  | grid cells filled: 2,761,240
  Processed 200,000,000 / 287,661,850 pts  | grid cells filled: 3,455,825
  Processed 240,000,000 / 287,661,850 pts  | grid cells filled: 4,157,624
  Processed 280,000,000 / 287,661,850 pts  | grid cells filled: 4,917,856
  Grid cells     : 5,093,418 filled / 10,678,365 total


In [ ]:
# CELL 9 | Process DHUNDA | Chunked loader | Uses trained XGBoost model

gc.collect()
dhunda_ground = classify_ground_chunked('DHUNDA', VILLAGES['DHUNDA'])
dhunda_dtm, dhunda_deriv, dhunda_vectors, _ = \
    process_village('DHUNDA', VILLAGES['DHUNDA'],
                    dhunda_ground, xgb_model, xgb_scaler)
print("DHUNDA complete")

NameError: name 'gc' is not defined

In [ ]:
# CELL 9B | Process KADAMTALA_RNG — Andaman, UTM 46N

gc.collect()
with laspy.open(VILLAGES['KADAMTALA_RNG']['raw']) as f:
    pts = f.header.point_count
print(f"KADAMTALA_RNG: {pts:,} points")

if pts > 50_000_000:
    ground = classify_ground_chunked('KADAMTALA_RNG', VILLAGES['KADAMTALA_RNG'])
else:
    ground = classify_ground_regular('KADAMTALA_RNG', VILLAGES['KADAMTALA_RNG'])

kada_dtm, kada_deriv, kada_vectors, _ = \
    process_village('KADAMTALA_RNG', VILLAGES['KADAMTALA_RNG'], ground)
print("KADAMTALA_RNG complete")

In [ ]:
# CELL 10 | Process DHAL | Regular loader | Uses trained XGBoost model

gc.collect()
dhal_ground = classify_ground_regular('DHAL', VILLAGES['DHAL'])
dhal_dtm, dhal_deriv, dhal_vectors, _ = \
    process_village('DHAL', VILLAGES['DHAL'],
                    dhal_ground, xgb_model, xgb_scaler)
print("DHAL complete")


[DHAL] Ground Classification (Regular)
  Raw points : 23,431,282
  Downsampled: 5,000,000
  Ground     : 4,623,222 (92.5%)
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/DHAL_ground.las

[DHAL] DTM Generation
  Grid       : 573 x 621 @ 1.0m
  Z range    : 0.00m -> 223.61m
  OGC COG    : /content/drive/MyDrive/mopr_outputs/dtm/DHAL_dtm_cog.tif

[DHAL] Terrain Derivatives
  Slope range  : 0.00 -> 0.97 deg
  Stream cells : 44,437

[DHAL] Rule labels  — High: 65,714 | Medium: 148,438

[DHAL] XGBoost
  Train labels   : {0: 79680, 1: 83180, 2: 37140}
  XGBoost trained: 200,000 px | 12 features
  Raw ML — High: 65,713 | Medium: 148,404

[DHAL] Morphological Post-processing
  Morphology: removed 7,172 noise pixels
  Final High : 61,608 | Medium: 145,337

[DHAL] Drainage Network Design
  Hotspot clusters : 6
  Proposed drains  : 4 | Total length: 0.35 km

[DHAL] Generating Village Report
  Report saved : /content/drive/MyDrive/mopr_outputs/village_reports/DHAL_report.png

[DHAL]

In [ ]:
# CELL 11 | Process THANDALAM | Processor chosen after Cell 4 inspection
# If pts > 50M: use classify_ground_chunked, else classify_ground_regular

gc.collect()
thand_pts = VILLAGES['THANDALAM']['pts']
print(f"THANDALAM point count: {thand_pts:,}")

if thand_pts > 50_000_000:
    thand_ground = classify_ground_chunked('THANDALAM', VILLAGES['THANDALAM'])
else:
    thand_ground = classify_ground_regular('THANDALAM', VILLAGES['THANDALAM'])

thand_dtm, thand_deriv, thand_vectors, _ = \
    process_village('THANDALAM', VILLAGES['THANDALAM'],
                    thand_ground, xgb_model, xgb_scaler)
print("THANDALAM complete")

THANDALAM point count: 188,077,336

[THANDALAM] Ground Classification (Chunked + NumPy Grid Accumulator)
  Total pts      : 188,077,336
  Coord type     : GEOGRAPHIC
  Voxel res      : 0.750 m  (0.00000693 native)
  Voxel grid     : 3129 x 2202 = 6.89M cells
  Stride         : 1/1  (effective pts: 188,077,336)
  Grid RAM       : 131.4 MB  (2202r × 3129c × 20 B/cell)
  Processed 40,000,000 / 188,077,336 pts  | grid cells filled: 727,364
  Processed 80,000,000 / 188,077,336 pts  | grid cells filled: 1,326,044
  Processed 120,000,000 / 188,077,336 pts  | grid cells filled: 1,960,679
  Processed 160,000,000 / 188,077,336 pts  | grid cells filled: 2,609,511
  Grid cells     : 3,115,598 filled / 6,890,058 total
  Post-cap pts   : 3,115,598
  Reprojected    : EPSG:32644
  Ground pts     : 2,475,090 (79.4%)
  Off-ground     : 640,508
  Saved LAS      : /content/drive/MyDrive/mopr_outputs/ground/THANDALAM_ground.las

[THANDALAM] DTM Generation
  Grid       : 2348 x 1693 @ 1.0m
  Z range    : -5

In [ ]:
# @title
# CELL 12 | Overview — transparent NODATA, cyan drain overlays

ALL_VILLAGES = ['DEVDI','KHAPRETA','PIRAYANKUPPAM','DHUNDA','DHAL',
                'THANDALAM','REFLIGHT_64334','CHAKHIRASINGH',
                'GANDHINAGAR_DIG','KADAMTALA_RNG']

available = {}
for vname in ALL_VILLAGES:
    dp = f'{OUT_BASE}/dtm/{vname}_dtm_cog.tif'
    rp = f'{OUT_BASE}/rasters/{vname}_risk_cog.tif'
    gp = f'{OUT_BASE}/gpkg/{vname}_drainage.gpkg'
    if os.path.exists(dp) and os.path.exists(rp):
        with rasterio.open(dp) as s:
            dtm  = s.read(1).astype(np.float32)
            tfm  = s.transform
        with rasterio.open(rp) as s:
            risk = s.read(1).astype(np.float32)
        drains = None
        if os.path.exists(gp):
            try:
                import fiona
                if 'proposed_drains' in fiona.listlayers(gp):
                    drains = gpd.read_file(gp, layer='proposed_drains')
            except: pass
        available[vname] = (dtm, risk, tfm, drains)
        print(f"  Loaded: {vname}  drains={len(drains) if drains is not None else 0}")

n   = len(available)
fig, axes = plt.subplots(2, n, figsize=(6*n, 12))
fig.patch.set_facecolor('#0d0d1a')
if n == 1: axes = axes.reshape(2,1)

for col, (vname, (dtm_arr, risk_arr, tfm, drains)) in enumerate(available.items()):

    # ── DTM row ──────────────────────────────────────────────────────────
    ax = axes[0][col]
    ax.set_facecolor('#0d0d1a')
    dtm_v = dtm_arr.copy(); dtm_v[dtm_v == NODATA] = np.nan
    valid  = dtm_v[~np.isnan(dtm_v)]
    im = ax.imshow(dtm_v, cmap='terrain', interpolation='bilinear',
                   vmin=np.percentile(valid,1), vmax=np.percentile(valid,99))
    ax.set_title(vname, color='white', fontsize=9, fontweight='bold', pad=3)
    ax.axis('off')
    plt.colorbar(im, ax=ax, label='m', shrink=0.7, pad=0.02)

    # ── Risk row ─────────────────────────────────────────────────────────
    ax2 = axes[1][col]
    ax2.set_facecolor('white')

    hs = np.round(risk_arr).astype(np.uint8)
    hs[dtm_arr == NODATA] = 255   # sentinel for transparent

    # Build RGBA — NODATA fully transparent
    rgba = np.zeros((*hs.shape, 4), dtype=np.float32)
    rgba[hs == 0]   = [0.95, 0.95, 0.95, 1.0]  # no risk — light grey
    rgba[hs == 1]   = [1.00, 0.55, 0.00, 1.0]  # medium — orange
    rgba[hs == 2]   = [0.80, 0.00, 0.00, 1.0]  # high   — red
    rgba[hs == 255] = [0.00, 0.00, 0.00, 0.0]  # NODATA — transparent

    ax2.imshow(rgba, interpolation='nearest')

    # Cyan drain overlay
    if drains is not None and len(drains) > 0:
        xmin_t = tfm.c; ymax_t = tfm.f; res_t = tfm.a
        for _, row_d in drains.iterrows():
            coords = list(row_d.geometry.coords)
            px_c   = [(c[0]-xmin_t)/res_t for c in coords]
            px_r   = [(ymax_t-c[1])/abs(tfm.e) for c in coords]
            ax2.plot(px_c, px_r, color='cyan', linewidth=1.2,
                     linestyle='--', alpha=0.85)

    n_high = int((hs==2).sum())
    n_med  = int((hs==1).sum())
    ax2.set_xlabel(f'H:{n_high:,} M:{n_med:,}',
                   color='black', fontsize=7)
    ax2.axis('off')

# Global legend
legend_handles = [
    mpatches.Patch(color='#FF8C00', label='Medium Risk'),
    mpatches.Patch(color='#CC0000', label='High Risk'),
    Line2D([0],[0], color='cyan', linestyle='--',
           linewidth=2, label='Proposed Drain'),
]
fig.legend(handles=legend_handles, loc='lower center',
           ncol=3, fontsize=10, facecolor='#1a1a2e',
           labelcolor='white', framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01))

fig.suptitle('DTM + Waterlogging Risk — IIT Tirupati MoPR Hackathon',
             fontsize=14, fontweight='bold', color='white')
plt.subplots_adjust(bottom=0.08)
out = f'{OUT_BASE}/reports/overview_final.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print(f"Saved: {out}")

In [ ]:
# CELL 12 | Verify all OGC-compliant outputs exist

print("=" * 65)
print("SUBMISSION PACKAGE CHECK")
print("=" * 65)
all_villages = ['DEVDI','KHAPRETA','PIRAYANKUPPAM','DHUNDA','DHAL','THANDALAM']
checks = []
for n in all_villages:
    checks += [
        (f'{n} Ground LAS',   f'{OUT_BASE}/ground/{n}_ground.las'),
        (f'{n} DTM COG',      f'{OUT_BASE}/dtm/{n}_dtm_cog.tif'),
        (f'{n} GPKG',         f'{OUT_BASE}/gpkg/{n}_drainage.gpkg'),
        (f'{n} Slope COG',    f'{OUT_BASE}/rasters/{n}_slope_cog.tif'),
        (f'{n} Risk COG',     f'{OUT_BASE}/rasters/{n}_risk_cog.tif'),
    ]
checks += [('Overview Report', f'{OUT_BASE}/reports/overview_final.png')]
all_ok = True
for label, path in checks:
    ok   = os.path.exists(path)
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if ok else "MISSING"
    print(f"  {'OK' if ok else 'XX'} {label:<30} {size}")
    if not ok: all_ok = False
print()
print("ALL FILES READY" if all_ok else "Some files missing — re-run failed cells")

SUBMISSION PACKAGE CHECK
  OK DEVDI Ground LAS               108.8 MB
  OK DEVDI DTM COG                  2.3 MB
  OK DEVDI GPKG                     1.1 MB
  OK DEVDI Slope COG                0.1 MB
  OK DEVDI Risk COG                 0.1 MB
  OK KHAPRETA Ground LAS            112.5 MB
  OK KHAPRETA DTM COG               1.4 MB
  OK KHAPRETA GPKG                  0.5 MB
  OK KHAPRETA Slope COG             0.1 MB
  OK KHAPRETA Risk COG              0.0 MB
  OK PIRAYANKUPPAM Ground LAS       118.8 MB
  OK PIRAYANKUPPAM DTM COG          5.6 MB
  OK PIRAYANKUPPAM GPKG             0.7 MB
  OK PIRAYANKUPPAM Slope COG        0.1 MB
  OK PIRAYANKUPPAM Risk COG         0.1 MB
  OK DHUNDA Ground LAS              116.5 MB
  OK DHUNDA DTM COG                 1.5 MB
  OK DHUNDA GPKG                    0.4 MB
  OK DHUNDA Slope COG               0.1 MB
  OK DHUNDA Risk COG                0.0 MB
  OK DHAL Ground LAS                120.2 MB
  OK DHAL DTM COG                   0.8 MB
  OK DHAL GPKG     